# Hybrid meta-cognition agent

This notebook keeps the demo logic local while importing the reusable `activation_steering` library components for planning, retrieval, steering, verification, and memory write-back.


In [ ]:
from pathlib import Path

from transformers import set_seed

import activation_steering as steering

set_seed(0)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LAYER_IDX = 12
TASK = "What is the capital of France, and what evidence supports that answer?"
ARTIFACT_PATH = Path("/tmp/hybrid_agent_feature_vectors.json")

model, tokenizer = steering.load_model_and_tokenizer(MODEL_NAME)
device = steering.get_model_device(model)


In [ ]:
feature_specs = steering.get_standard_feature_specs()
discovered_vectors = steering.discover_and_store_feature_vectors(
    feature_specs=feature_specs,
    layer_idx=LAYER_IDX,
    model=model,
    tokenizer=tokenizer,
    device=device,
    output_path=ARTIFACT_PATH,
)

controllers = steering.load_steering_controllers(
    ARTIFACT_PATH,
    default_alpha=1.0,
    default_decay=0.9,
    task_types_by_controller={
        "retrieval_augmented_context": ["qa"],
        "chain_of_thought": ["qa"],
    },
)

memory = steering.InMemorySteeringMemory(controllers)
executor = steering.SteeredExecutor(
    model=model,
    tokenizer=tokenizer,
    device=device,
    model_name=MODEL_NAME,
)

print(f"Persisted {len(discovered_vectors)} feature vectors to {ARTIFACT_PATH}")
print("Controllers:", [controller.controller_id for controller in controllers])


In [ ]:
def planner(task: str, memory_store: steering.InMemorySteeringMemory) -> steering.PlannerDecision:
    needs_retrieval = "evidence" in task.lower() or "support" in task.lower()
    return steering.PlannerDecision(
        task_type="qa",
        needs_retrieval=needs_retrieval,
        controller_id="retrieval_augmented_context" if needs_retrieval else "chain_of_thought",
        reasoning_effort="medium",
        use_steering=True,
        allow_fallback=True,
    )


def retriever(task: str, plan: steering.PlannerDecision) -> list[str]:
    return [
        "Reference: Paris is the capital and most populous city of France.",
        "Reference: France is a country in Western Europe.",
    ]


def tool_router(task: str, plan: steering.PlannerDecision, context: list[str]) -> list[str]:
    return ["Tool observation: retrieved references are sufficient for a grounded short answer."]


def verifier(
    task: str,
    draft: steering.ExecutorResult,
    context: list[str],
    plan: steering.PlannerDecision,
) -> steering.VerifierResult:
    has_context = bool(context)
    return steering.VerifierResult(
        passed=has_context and len(draft.output_text) > 0,
        confidence=0.8 if has_context else 0.3,
        issues=[] if has_context else ["missing retrieval evidence"],
    )


In [ ]:
agent = steering.HybridMetaCognitionAgent(
    planner=planner,
    retriever=retriever,
    tool_router=tool_router,
    executor=executor,
    verifier=verifier,
    memory=memory,
    max_new_tokens=40,
)

run = agent.run(TASK)

print("Selected controller:", run.selected_controller_id)
print("Fallback used:", run.fallback_used)
print("Verifier confidence:", run.verdict.confidence)
print("Top trace features:", [(score.feature_id, round(score.score, 3)) for score in run.draft.activation_trace.top_feature_scores])
print("\nOutput:\n", run.draft.output_text)
